<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# Bayesian Personalized Ranking (BPR)

This notebook serves as an introduction to Bayesian Personalized Ranking (BPR) model for implicit feedback.  In this tutorial, we focus on learning the BPR model using matrix factorization approach, hence, the model is sometimes also named BPRMF.

The implementation of the model is from [Cornac](https://github.com/PreferredAI/cornac), which is a framework for recommender systems with a focus on models leveraging auxiliary data (e.g., item descriptive text and image, social network, etc).

## 0 Global Settings and Imports

In [1]:
import sys
import cornac

from recommenders.datasets import movielens
from recommenders.datasets.python_splitters import python_random_split
from recommenders.evaluation.python_evaluation import map_at_k, ndcg_at_k, precision_at_k, recall_at_k
from recommenders.models.cornac.bpr import BPR
from recommenders.utils.timer import Timer
from recommenders.utils.constants import SEED
from recommenders.utils.python_utils import binarize
from recommenders.utils.notebook_utils import store_metadata

print(f"System version: {sys.version}")
print(f"Cornac version: {cornac.__version__}")

/home/u/.venvs/recommenders/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


System version: 3.11.14 (main, Jan 14 2026, 19:35:32) [Clang 21.1.4 ]
Cornac version: 2.3.0


In [2]:
# Select MovieLens data size: 100k, 1m, 10m, or 20m
MOVIELENS_DATA_SIZE = "100k"

# top k items to recommend
TOP_K = 10

# Minimum rating to consider as positive implicit feedback
RATING_THRESHOLD = 3.5

# Model parameters
NUM_FACTORS = 200
NUM_EPOCHS = 200

## 1 BPR Algorithm

### 1.1 Personalized Ranking from Implicit Feedback

The task of personalized ranking aims at providing each user a ranked list of items (recommendations). This is very common in scenarios where recommender systems are based on implicit user behavior (e.g. purchases, clicks). The available observations are only positive feedback where the non-observed ones are a mixture of real negative feedback and missing values.

One usual approach for item recommendation is directly predicting a preference score $\hat{x}_{u,i}$ given to item $i$ by user $u$. This method uses the **pointwise loss** to optimize the model. Methods based on pointwise learning usually follow a regression framework by minimizing the  squared loss between $\hat{x}_{u,i}$ its target value ${x}_{u,i}$. Unfortunately, this method doesn't work well with ranking.

<p align="center">
<img src="https://raw.githubusercontent.com/recommenders-team/resources/main/images/bpr1.png" width="400px">
</p>

In the implicit feedback scenario the observed data is positive feedback. As it is shown in the figure above, the observed data (image to the left) is known, but the unobserved data is unknown. The negative data is usually generated by filling the matrix with 0 (image to the right). 

Using the pointwise loss, the model is trained to predict as 1 the elements in the observed dataset and 0 for the rest. The problem with this approach is that all elements the model should rank in the future are presented as negative feedback during training but in reality, some of them are unknown items and some of them are true negatives. 

BPR uses a different approach by using item pairs $(i, j)$ and optimizing for the correct ranking given preference of user $u$, thus, there are notions of *positive* and *negative* items. The training data $D_S : U \times I \times I$ is defined as:

$$D_S = \{(u, i, j) \mid i \in I^{+}_{u} \wedge j \in I \setminus I^{+}_{u}\}$$

where user $u$ is assumed to prefer $i$ over $j$ (i.e. $i$ is a *positive item* and $j$ is a *negative item*).

This approach uses the **pairwise loss**. The intuition behind it is that observed entries should be ranked higher than the unobserved ones. Then instead of minimizing the loss between $\hat{x}_{u,i}$ and ${x}_{u,i}$, pairwise learning maximizes the margin between the observed entry $\hat{x}_{u,i}$ and the unobserved entry $\hat{x}_{u,j}$.

<p align="center">
<img src="https://raw.githubusercontent.com/recommenders-team/resources/main/images/bpr2.png" width="400px">
</p>

In the figure above, pairwise learning is explained. For each user, a pairwise matrix of items is generated. The plus indicates that the user prefers item *i* over item *j*, and the minus is that the user prefers item *j* over *i*. The interrogation indicates that we don't know the user's preference.

The pairwise setup has several advantages:
* The training set is defined as positive, negative and unknown preferences. 
* The missing values between two non-observed items are exactly the item pairs that have to be ranked by the model.
* The training data is created for the actual objective of ranking.
* Pairwise ranking is not restricted to implicit feedback and can be used in explicit feedback by setting user preferences as pairs (i.e. user *u* prefers item *a* more than item *b*).

### 1.2 Pairwise Objective Function

From the Bayesian perspective, BPR maximizes the posterior probability over the model parameters $\Theta$ by optimizing the likelihood function $p(i >_{u} j | \Theta)$ and the prior probability $p(\Theta)$.

$$p(\Theta \mid >_{u}) \propto p(i >_{u} j \mid \Theta) \times p(\Theta)$$

The joint probability of the likelihood over all users $u \in U$ can be simplified to:

$$ \prod_{u \in U} p(>_{u} \mid \Theta) = \prod_{(u, i, j) \in D_S} p(i >_{u} j \mid \Theta) $$

The individual probability that a user $u$ prefers item $i$ to item $j$ can be defined as:

$$ p(i >_{u} j \mid \Theta) = \sigma (\hat{x}_{uij}(\Theta)) $$

where $\sigma$ is the logistic sigmoid:

$$ \sigma(x) = \frac{1}{1 + e^{-x}} $$

The preference scoring function $\hat{x}_{uij}(\Theta)$ could be an arbitrary real-valued function of the model parameter $\Theta$.  Thus, it makes BPR a general framework for modeling the relationship between triplets $(u, i, j)$ where different model classes like matrix factorization could be used for estimating $\hat{x}_{uij}(\Theta)$.

For the prior, one of the common pratices is to choose $p(\Theta)$ following a normal distribution, which results in a nice form of L2 regularization in the final log-form of the objective function.

$$ p(\Theta) \sim N(0, \Sigma_{\Theta}) $$

To reduce the complexity of the model, all parameters $\Theta$ are assumed to be independent and having the same variance, which gives a simpler form of the co-variance matrix $\Sigma_{\Theta} = \lambda_{\Theta}I$.  Thus, there are less number of hyperparameters to be determined.

The final objective of the maximum posterior estimator:

$$ J = \sum_{(u, i, j) \in D_S} \text{ln } \sigma(\hat{x}_{uij}) - \lambda_{\Theta} ||\Theta||^2 $$

where $\lambda_\Theta$ are the model specific regularization parameters.


### 1.3 Learning with Matrix Factorization

#### Stochastic Gradient Descent

As the defined objective function is differentiable, gradient descent based method for optimization is naturally adopted.  The gradient of the objective $J$ with respect to the model parameters:

$$
\begin{align}
\frac{\partial J}{\partial \Theta} & = \sum_{(u, i, j) \in D_S} \frac{\partial}{\partial \Theta} \text{ln} \ \sigma(\hat{x}_{uij}) - \lambda_{\Theta} \frac{\partial}{\partial \Theta} ||\Theta||^2 \\
& \propto \sum_{(u, i, j) \in D_S} \frac{-e^{-\hat{x}_{uij}}}{1 + e^{-\hat{x}_{uij}}} \cdot  \frac{\partial}{\partial \Theta} \hat{x}_{uij} - \lambda_{\Theta} \Theta
\end{align}
$$

Due to slow convergence of full gradient descent, we prefer using stochastic gradient descent to optimize the BPR model.  For each triplet $(u, i, j) \in D_S$, the update rule for the parameters:

$$ \Theta \leftarrow \Theta + \alpha \Big( \frac{e^{-\hat{x}_{uij}}}{1 + e^{-\hat{x}_{uij}}} \cdot \frac{\partial}{\partial \Theta} \hat{x}_{uij} + \lambda_\Theta \Theta \Big) $$

#### Matrix Factorization for Preference Approximation

As mentioned earlier, the preference scoring function $\hat{x}_{uij}(\Theta)$ could be approximated by any real-valued function.  First, the estimator $\hat{x}_{uij}$ is decomposed into:

$$ \hat{x}_{uij} = \hat{x}_{ui} - \hat{x}_{uj} $$

The problem of estimating $\hat{x}_{ui}$ is a standard collaborative filtering formulation, where matrix factorization approach has shown to be very effective.  The prediction formula can written as dot product between user feature vector $w_u$ and item feature vector $h_i$:

$$ \hat{x}_{ui} = \langle w_u , h_i \rangle = \sum_{f=1}^{k} w_{uf} \cdot h_{if} $$

The  derivatives of matrix factorization with respect to the model parameters are:

$$
\frac{\partial}{\partial \theta} \hat{x}_{uij} = 
\begin{cases}
    (h_{if} - h_{jf})  & \text{if } \theta = w_{uf} \\
    w_{uf}             & \text{if } \theta = h_{if} \\
    -w_{uf}            & \text{if } \theta = h_{jf} \\
    0                  & \text{else}
\end{cases}
$$

In theory, any kernel can be used to estimate $\hat{x}_{ui}$ besides the dot product $ \langle \cdot , \cdot \rangle $.  For example, k-Nearest-Neighbor (kNN) has also been shown to achieve good performance.

#### Analogies to AUC optimization

By optimizing the objective function of BPR model, we effectively maximizing [AUC](https://towardsdatascience.com/understanding-auc-roc-curve-68b2303cc9c5) measurement. To keep the notebook focused, please refer to the [paper](https://arxiv.org/ftp/arxiv/papers/1205/1205.2618.pdf) for details of the analysis (Section 4.1.1).

## 2 Cornac implementation of BPR

BPR is implemented in the [Cornac](https://cornac.readthedocs.io/en/latest/index.html) framework as part of the model collections.
* Detailed documentations of the BPR model in Cornac can be found [here](https://cornac.readthedocs.io/en/latest/models.html#bayesian-personalized-ranking-bpr).
* Source codes of the BPR implementation is available on the Cornac Github repository, which can be found [here](https://github.com/PreferredAI/cornac/blob/master/cornac/models/bpr/recom_bpr.pyx).


## 3 Cornac BPR movie recommender


### 3.1 Load and split data

To evaluate the performance of item recommendation, we adopted the provided `python_random_split` tool for the consistency.  Data is randomly split into training and test sets with the ratio of 75/25.


Note that Cornac also cover different [built-in schemes](https://cornac.readthedocs.io/en/latest/eval_methods.html) for model evaluation.

In [3]:
data = movielens.load_pandas_df(
    size=MOVIELENS_DATA_SIZE,
    header=["userID", "itemID", "rating"]
)


data['rating'] = binarize(data['rating'].values, RATING_THRESHOLD)
data = data[data['rating'] > 0].reset_index(drop=True)
data.head()

  0%|          | 0.00/4.81k [00:00<?, ?KB/s]

  0%|          | 8.00/4.81k [00:00<01:21, 58.8KB/s]

  1%|          | 39.0/4.81k [00:00<00:31, 152KB/s] 

  2%|▏         | 94.0/4.81k [00:00<00:18, 259KB/s]

  4%|▎         | 172/4.81k [00:00<00:12, 372KB/s] 

...

 32%|███▏      | 1.54k/4.81k [00:01<00:01, 2.33kKB/s]

 49%|████▊     | 2.34k/4.81k [00:01<00:00, 3.22kKB/s]

 55%|█████▌    | 2.66k/4.81k [00:01<00:00, 2.99kKB/s]

100%|██████████| 4.81k/4.81k [00:01<00:00, 3.25kKB/s]

,userID,itemID,rating
0,298,474,1.0
1,253,465,1.0
2,286,1014,1.0
3,200,222,1.0
4,122,387,1.0


In [4]:
train, test = python_random_split(data, 0.75)

### 3.2 Cornac Dataset

To work with models implemented in Cornac, we need to construct an object from [Dataset](https://cornac.readthedocs.io/en/latest/data.html#module-cornac.data.dataset) class.

Dataset Class in Cornac serves as the main object that the models will interact with.  In addition to data transformations, Dataset provides a bunch of useful iterators for looping through the data, as well as supporting different negative sampling techniques.

In [5]:
train_set = cornac.data.Dataset.from_uir(train.itertuples(index=False), seed=SEED)

print('Number of users: {}'.format(train_set.num_users))
print('Number of items: {}'.format(train_set.num_items))

Number of users: 942
Number of items: 1395


### 3.3 Train the BPR model

The BPR has a few important parameters that we need to consider:

- `k`: controls the dimension of the latent space (i.e. the size of the vectors  $w_u$  and  $h_i$ ).
- `max_iter`: defines the number of iterations of the SGD procedure.
- `learning_rate`: controls the step size $\alpha$ in the gradient update rules.
- `lambda_reg`: controls the L2-Regularization $\lambda$ in the objective function.

Note that different values of `k` and `max_iter` will affect the training time.

We will here set `k` to 200, `max_iter` to 100, `learning_rate` to 0.01, and `lambda_reg` to 0.001. To train the model, we simply need to call the `fit()` method.

In [6]:
bpr = BPR(
    k=NUM_FACTORS,
    max_iter=NUM_EPOCHS,
    learning_rate=0.01,
    lambda_reg=0.001,
    verbose=True,
    seed=SEED
)

In [7]:
with Timer() as t:
    bpr.fit(train_set)
print(f"Took {t} seconds for training.")

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s, correct=82.01%, skipped=6.02%]

  0%|          | 0/200 [00:00<?, ?it/s, correct=84.00%, skipped=5.68%]

  0%|          | 0/200 [00:00<?, ?it/s, correct=84.58%, skipped=5.88%]

  0%|          | 0/200 [00:00<?, ?it/s, correct=84.13%, skipped=5.93%]

...

 98%|█████████▊| 195/200 [00:04<00:00, 45.18it/s, correct=95.91%, skipped=6.08%]

 98%|█████████▊| 195/200 [00:04<00:00, 45.18it/s, correct=96.13%, skipped=5.68%]

100%|██████████| 200/200 [00:04<00:00, 45.17it/s, correct=96.13%, skipped=5.68%]

100%|██████████| 200/200 [00:04<00:00, 44.91it/s, correct=96.13%, skipped=5.68%]

Optimization finished!
Took 4.4787 seconds for training.


### 3.4 Prediction and Evaluation

Now that our model is trained, we can produce the ranked lists for recommendation.  Every recommender models in Cornac provide `rate()` and `rank()` methods for predicting item rated value as well as item ranked list for a given user.  To make use of the current evaluation schemes, we will through `predict()` and `predict_ranking()` functions inside `cornac_utils` to produce the predictions.

Note that BPR model is effectively designed for item ranking.  Hence, we only measure the performance using ranking metrics.

#### Understanding the prediction scores

The `prediction` column contains the raw dot product score $\hat{x}_{ui} = \langle w_u, h_i \rangle$ between the user and item latent vectors. These scores are **unbounded** — they can be any real number and are typically larger than 1 when `k` is large (200 factors here). This is expected and correct.

BPR never learns to predict a value in $[0, 1]$. During training, the sigmoid is applied to the **difference** of two scores $\sigma(\hat{x}_{ui} - \hat{x}_{uj})$, not to individual scores. The absolute magnitude of each score is therefore arbitrary; only the **relative ordering** across items matters for ranking.

In [8]:
with Timer() as t:
    all_predictions = bpr.recommend_k_items(train, col_user='userID', col_item='itemID', remove_seen=True)
print(f"Took {t} seconds for prediction.")

Took 0.5332 seconds for prediction.


In [9]:
all_predictions.head()

,userID,itemID,prediction
0,925,313,4.508990
1,925,258,4.431099
2,925,127,4.348322
3,925,100,4.343854
4,925,50,4.340135


In [10]:
eval_map = map_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)
eval_ndcg = ndcg_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)
eval_precision = precision_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)
eval_recall = recall_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)


print(f"Model:",
      f"Top K:\t\t {TOP_K}",
      f"MAP@K:\t\t {eval_map:f}",
      f"NDCG@K:\t\t {eval_ndcg:f}",
      f"Precision@K:\t {eval_precision:f}",
      f"Recall@K:\t {eval_recall:f}", sep="\n")

Model:
Top K:		 10
MAP@K:		 0.180368
NDCG@K:		 0.305190
Precision@K:	 0.239032
Recall@K:	 0.208660


In [11]:
# Record results for tests - ignore this cell
store_metadata("map", eval_map)
store_metadata("ndcg", eval_ndcg)
store_metadata("precision", eval_precision)
store_metadata("recall", eval_recall)

## References

1. Rendle, S., Freudenthaler, C., Gantner, Z., & Schmidt-Thieme, L. (2009, June). BPR: Bayesian personalized ranking from implicit feedback. https://arxiv.org/ftp/arxiv/papers/1205/1205.2618.pdf
2. Pan, R., Zhou, Y., Cao, B., Liu, N. N., Lukose, R., Scholz, M., & Yang, Q. (2008, December). One-class collaborative filtering. https://cseweb.ucsd.edu/classes/fa17/cse291-b/reading/04781145.pdf
3. **Cornac** - A Comparative Framework for Multimodal Recommender Systems. https://cornac.preferred.ai/